# Howard County, MD: County + Fire & Rescue Levy LVT Model

This notebook models a revenue-neutral land value tax shift for **Howard County, Maryland** at a
**4:1 land-to-improvement millage ratio**, holding the county's existing exemptions, assessment
phase-in and Homestead Tax Credit in place.

## Policy assumptions (confirmed up front)

- **Scope**: the two real-property levies Howard County itself sets, the **county general property
  tax** and the **Fire & Rescue Tax**. Howard has no incorporated municipalities (Columbia and
  Ellicott City are unincorporated), so there is no city layer, and the school system is funded out
  of the county general levy rather than by a separate school tax. The Maryland state property tax
  and the Metropolitan District ad valorem charge also appear on Howard bills and are **not**
  modeled.
- **Reform**: revenue-neutral split-rate at **4:1** (land taxed at 4x the improvement rate), solved
  once across both levies. Both are countywide flat rates on the same assessed base, so a single
  solve and two separate solves give identical parcel-level results.
- **Exemptions and credits preserved**: full exemptions stay exempt, partial exemptions are kept at
  their billed amount, and each parcel's phase-in and Homestead credit carry over (see Section 4).
- **Assessment ratio**: Maryland assesses at 100% of market value, phased in over the three-year
  cycle.
- **Tax year**: FY 2027 (levy year beginning July 1, 2026), the year SDAT's current roll describes.

## Data

- **Assessment roll**: Maryland Real Property Assessments, Maryland Open Data (Socrata dataset
  `ed4q-f8tm`), filtered to `HOWA`. Unlike the iMAP parcel layer, it carries the *billed*
  quantities: the current-year phase-in assessment, the county exempt assessment and the county
  Homestead assessment credit.
- **Geometry**: MD iMAP `PlanningCadastre/MD_ParcelBoundaries/MapServer/0`, joined on `ACCTID`.

## Section 1: Imports and constants

In [1]:
import sys
import os
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from shapely.geometry import Point, Polygon
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

# Constants
CITY_NAME = 'howard_county'
STATE_FIPS = '24'              # Maryland
COUNTY_FIPS = '027'            # Howard County
MODEL_TYPE = 'split_rate:4.0'
LAND_IMPROVEMENT_RATIO = 4.0

# FY 2027 real-property rates per $100 of assessment (Howard County Finance, "Real Property Tax",
# effective July 1, 2026; unchanged from FY 2026 per the FY 2027 Approved Budget).
COUNTY_RATE_PER_100 = 1.044
FIRE_RATE_PER_100 = 0.206
COUNTY_MILLAGE = COUNTY_RATE_PER_100 * 10.0     # per $1,000
FIRE_MILLAGE = FIRE_RATE_PER_100 * 10.0
COMBINED_MILLAGE = COUNTY_MILLAGE + FIRE_MILLAGE

# Parcel-map export identity columns
PARCEL_ID_COL = 'acctid'
OWNER_NAME_COL = None            # the public roll hides owner names
OWNER_ADDRESS_COL = 'address'    # situs address
# SDAT's detail page needs the account split into district + number, which a {parcel_id}
# template cannot express; the per-account link built in Section 2 is attached in Section 7.
PARCEL_URL_TEMPLATE = None

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

C:\Users\druss\miniconda3\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Section 2: Fetch / load parcel data

Two sources, joined on the SDAT account id.

**Assessment roll.** `opendata.maryland.gov` now answers scripted requests with a Cloudflare
challenge (HTTP 403), so the cached CSV was saved from a browser session using the query that
`SDAT_EXPORT_URL` builds below (only the columns used here, `HOWA` only). If the cache is absent
the cell tries a plain request and, on a 403, stops with the URL to open in a browser.

**Geometry.** The iMAP parcel layer is a `MapServer` with a 1,000-record page limit and accepts a
`where` clause, so an inline fetcher (after the Rockville notebook) pages through `JURSCODE='HOWA'`.

In [2]:
from urllib.parse import urlencode

SDAT_CACHE = DATA_DIR / 'sdat_howa_2026-09-23.csv'
SDAT_FIELDS = {
    'account_id_mdp_field_acctid': 'acctid',
    'record_key_district_ward_sdat_field_2': 'district',
    'record_key_owner_occupancy_code_mdp_field_ooi_sdat_field_6': 'owner_occ',
    'mdp_longitude_mdp_field_digxcord_converted_to_wgs84': 'lon',
    'mdp_latitude_mdp_field_digycord_converted_to_wgs84': 'lat',
    'mdp_street_address_mdp_field_address': 'address',
    'mdp_street_address_city_mdp_field_city': 'city',
    'premise_address_condominium_unit_no_sdat_field_28': 'condo_unit',
    'town_code_mdp_field_towncode_desctown_sdat_field_36': 'town',
    'exempt_class_mdp_field_exclass_descexcl_sdat_field_49': 'exempt_class',
    'land_use_code_mdp_field_lu_desclu_sdat_field_50': 'land_use',
    'multi_parent_account_ind_sdat_field_55': 'multi_parent',
    'county_system_property_code_sdat_field_56': 'county_prop_code',
    'county_service_code_sdat_field_57': 'county_service_code',
    'tax_class_sdat_field_58': 'tax_class',
    'bpruc_public_use_code_mdp_field_ciuse_descciuse_sdat_field_61': 'ciuse',
    'ad_valorem_code_sdat_field_62': 'ad_valorem_code',
    'full_and_partial_exemptions_county_exempt_class_sdat_field_139': 'cty_exempt_class',
    'full_and_partial_exemptions_county_exempt_assessment_sdat_field_140': 'cty_exempt_assmt',
    'full_and_partial_exemptions_county_exempt_percentage_sdat_field_141': 'cty_exempt_pct',
    'full_and_partial_exemptions_state_exempt_assessment_sdat_field_143': 'sta_exempt_assmt',
    'base_cycle_data_land_value_sdat_field_154': 'base_land',
    'base_cycle_data_improvements_value_sdat_field_155': 'base_impr',
    'prior_assessment_year_total_assessment_sdat_field_161': 'prior_total_assmt',
    'current_cycle_data_land_value_mdp_field_names_nfmlndvl_curlndvl_and_sallndvl_sdat_field_164': 'cur_land',
    'current_cycle_data_improvements_value_mdp_field_names_nfmimpvl_curimpvl_and_salimpvl_sdat_field_165': 'cur_impr',
    'current_cycle_data_preferential_land_value_sdat_field_166': 'cur_pref_land',
    'current_assessment_year_total_phase_in_value_sdat_field_171': 'cur_phase_in',
    'current_assessment_year_total_assessment_sdat_field_172': 'cur_total_assmt',
    'assessment_credit_program_current_state_assmt_cr_sdat_field_197': 'sta_assmt_credit',
    'assessment_credit_program_current_county_assmt_cr_sdat_field_199': 'cty_assmt_credit',
    'assessment_credit_program_current_credit_status_code_sdat_field_202': 'credit_status',
    'homestead_qualification_code_mdp_field_homqlcod_sdat_field_259': 'homestead_code',
    'homestead_qualification_date_mdp_field_homqldat_sdat_field_260': 'homestead_date',
    'residential_exception_indicator_sdat_field_261': 'res_exception',
    'c_a_m_a_system_data_year_built_yyyy_mdp_field_yearblt_sdat_field_235': 'year_built',
    'c_a_m_a_system_data_number_of_dwelling_units_mdp_field_bldg_units_sdat_field_239': 'units',
    'c_a_m_a_system_data_number_of_stories_mdp_field_bldg_story_sdat_field_240': 'stories',
    'c_a_m_a_system_data_structure_area_sq_ft_mdp_field_sqftstrc_sdat_field_241': 'struct_area',
    'c_a_m_a_system_data_land_area_mdp_field_landarea_sdat_field_242': 'land_area',
    'c_a_m_a_system_data_land_unit_of_measure_mdp_field_luom_sdat_field_243': 'land_uom',
    'additional_c_a_m_a_data_building_style_code_and_description_mdp_field_strustyl_descstyl_sdat_field_264': 'style',
    'additional_c_a_m_a_data_dwelling_type_mdp_field_strubldg_sdat_field_265': 'dwel_type',
    'parent_account_number_account_number_sdat_field_388': 'parent_acct',
    'record_deletion_date_yyyy_mm_dd_sdat_field_398': 'deletion_date',
    'assessment_cycle_year_sdat_field_399': 'cycle_year',
    'file_record_type_sdat_field_400': 'record_type',
    'date_of_most_recent_open_data_portal_record_update': 'odp_update',
}
SDAT_EXPORT_URL = 'https://opendata.maryland.gov/resource/ed4q-f8tm.csv?' + urlencode({
    '$select': ', '.join(f'{k} as {v}' for k, v in SDAT_FIELDS.items()),
    '$where': "jurisdiction_code_mdp_field_jurscode='HOWA'",
    '$order': 'account_id_mdp_field_acctid',
    '$limit': 200000,
})

if not SDAT_CACHE.exists():
    r = requests.get(SDAT_EXPORT_URL, timeout=1800)
    if r.status_code == 403:
        raise RuntimeError(
            'opendata.maryland.gov returned a Cloudflare challenge. Open this URL in a browser and '
            f'save the CSV as {SDAT_CACHE}:\n{SDAT_EXPORT_URL}')
    r.raise_for_status()
    SDAT_CACHE.write_bytes(r.content)

sdat = pd.read_csv(SDAT_CACHE, dtype=str)
print(f"SDAT roll rows (HOWA): {len(sdat):,}   unique accounts: {sdat['acctid'].nunique():,}")
print("Assessment cycle year:", sdat['cycle_year'].value_counts().to_dict())

SDAT roll rows (HOWA): 113,014   unique accounts: 113,014
Assessment cycle year: {'2027': 112796, '2023': 42, '2025': 40, '2022': 38, '2024': 35, '2026': 25, '2021': 20, '2020': 17, '2019': 1}


In [3]:
PARCEL_QUERY_URL = (
    'https://mdgeodata.md.gov/imap/rest/services/'
    'PlanningCadastre/MD_ParcelBoundaries/MapServer/0/query'
)
IMAP_WHERE = "JURSCODE='HOWA'"
IMAP_FIELDS = 'OBJECTID,ACCTID,LU,DESCLU,CIUSE,DESCCIUSE,DESCSTYL,DESCBLDG,BLDG_UNITS,SQFTSTRC,SDATWEBADR,POLYACRES,PTYPE'
POLY_CACHE = DATA_DIR / 'howard_parcels_imap.gpq'


def _esri_rings_to_shapely(geom):
    """ArcGIS rings -> shapely; clockwise rings are shells, counter-clockwise rings are holes."""
    rings = (geom or {}).get('rings') or []
    if not rings:
        return None
    shells, holes = [], []
    for ring in rings:
        p = Polygon(ring)
        (holes if p.exterior.is_ccw else shells).append(p.buffer(0))
    out = None
    for shell in shells:
        for hole in holes:
            if shell.contains(hole.representative_point()):
                shell = shell.difference(hole)
        out = shell if out is None else out.union(shell)
    return out


def fetch_md_parcels(query_url, where, fields, chunk_size=1000):
    session = requests.Session()
    count = session.get(query_url, params={'f': 'json', 'where': where, 'returnCountOnly': 'true'},
                        timeout=60).json()['count']
    print(f"Total matching polygons: {count:,}")
    rows = []
    for offset in range(0, count, chunk_size):
        params = {'f': 'json', 'where': where, 'outFields': fields, 'returnGeometry': 'true',
                  'outSR': 4326, 'geometryPrecision': 6, 'resultOffset': offset,
                  'resultRecordCount': chunk_size, 'orderByFields': 'OBJECTID ASC'}
        for attempt in range(4):
            try:
                r = session.get(query_url, params=params, timeout=180)
                r.raise_for_status()
                feats = r.json()['features']
                break
            except Exception as e:
                print(f"  retry at offset {offset}: {e}")
        for f in feats:
            attrs = f['attributes']
            attrs['geometry'] = _esri_rings_to_shapely(f.get('geometry'))
            rows.append(attrs)
    return gpd.GeoDataFrame(pd.DataFrame(rows), geometry='geometry', crs='EPSG:4326')


if POLY_CACHE.exists():
    polys = gpd.read_parquet(POLY_CACHE)
    print(f"Loaded {len(polys):,} iMAP polygons from cache")
else:
    polys = fetch_md_parcels(PARCEL_QUERY_URL, IMAP_WHERE, IMAP_FIELDS)
    polys.to_parquet(POLY_CACHE)
    print(f"Downloaded and cached {len(polys):,} iMAP polygons")

Loaded 100,460 iMAP polygons from cache


### Join roll to geometry

The roll is the unit of analysis (one row per SDAT account, the unit that is billed). About 87% of
current accounts have their own iMAP polygon. The rest are chiefly condominium units, which share
their building's footprint, and townhouses in recent subdivisions not yet digitized; they keep the
roll's own point location (SDAT `DIGXCORD/DIGYCORD`) drawn as a small circle, so census matching
still uses the true location. No roll account is dropped for lacking a polygon.

In [4]:
polys_j = polys.dropna(subset=['ACCTID']).drop_duplicates('ACCTID')
gdf = sdat.merge(polys_j.drop(columns=['OBJECTID']), left_on='acctid', right_on='ACCTID', how='left')
has_poly = gdf['geometry'].notna()

lon = pd.to_numeric(gdf['lon'], errors='coerce')
lat = pd.to_numeric(gdf['lat'], errors='coerce')
pts = gpd.GeoSeries([Point(x, y) if pd.notna(x) and pd.notna(y) else None for x, y in zip(lon, lat)],
                    crs='EPSG:4326')
# 6 m circles for accounts without their own polygon (drawn in a metric CRS, then back to WGS84)
circles = pts.to_crs('EPSG:26985').buffer(6).to_crs('EPSG:4326')
gdf['geometry'] = gdf['geometry'].where(has_poly, circles)
gdf['geom_source'] = np.where(has_poly, 'imap_polygon', np.where(pts.notna(), 'roll_point', 'none'))
gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs='EPSG:4326')

# SDAT detail link, built for every account (county code 14 + district + 6-digit account)
gdf['SDATWEBADR'] = ('https://sdat.dat.maryland.gov/RealProperty/Pages/viewdetails.aspx?County=14'
                     '&SearchType=ACCT&District=' + gdf['district'].str.zfill(2)
                     + '&AccountNumber=' + gdf['acctid'].str[-6:])

print(gdf['geom_source'].value_counts().to_string())
print("Accounts without polygon, by land use:")
print(gdf.loc[~has_poly, 'land_use'].value_counts().head(8).to_string())

geom_source
imap_polygon    97745
roll_point      14201
none             1068
Accounts without polygon, by land use:
land_use
Residential Condominium (U)    7084
Town House (TH)                5563
Commercial Condominium (CC)    1061
Exempt (E)                      666
Residential (R)                 635
Exempt Commercial (EC)          145
Commercial (C)                   59
Apartments (M)                   29


### Column mapping

| Concept | Column | Notes |
|---|---|---|
| Land value (full market) | `cur_land` | SDAT field 164, current-cycle appraisal |
| Improvement value (full market) | `cur_impr` | SDAT field 165 |
| Billed assessment before exemptions and credits | `cur_total_assmt` | SDAT field 172 = phase-in value (171) for every current record |
| County exempt assessment | `cty_exempt_assmt` | SDAT field 140; equals the whole assessment for fully exempt parcels |
| County Homestead assessment credit | `cty_assmt_credit` | SDAT field 199; the 5% county Homestead cap, in assessment dollars |
| Use code | `land_use` | SDAT field 50 (R, TH, U, C, CC, I, M, A, CR, RC, CA, E, EC) |
| Commercial subtype | `DESCCIUSE` | from iMAP (description of SDAT field 61) |
| Units / structure area | `units`, `struct_area` | CAMA fields 239 / 241 |
| Parcel ID | `acctid` | SDAT account id (county 14 + district + account) |

## Section 3: Classify and validate

1. Drop the stale records (assessment cycle other than 2027: deleted or superseded accounts with no
   current value) and the $0-value accounts (common areas, slivers).
2. Coerce values; flag **full exemption off the billed quantity**: a parcel is fully exempt when its
   county exempt assessment covers its whole assessment, not because of the recorded exempt class
   (the class field and the exempt assessment disagree on a number of accounts in both directions).
3. Map every parcel to a `PROPERTY_CATEGORY`; $0 improvement -> `Vacant Land`.

In [5]:
n_raw = len(gdf)
gdf = gdf[gdf['cycle_year'] == '2027'].copy()
n_current = len(gdf)
print(f"Dropped {n_raw - len(gdf):,} stale records (assessment cycle before 2027); {len(gdf):,} current accounts")

NUM_COLS = ['cur_land', 'cur_impr', 'cur_pref_land', 'cur_phase_in', 'cur_total_assmt',
            'cty_exempt_assmt', 'cty_exempt_pct', 'cty_assmt_credit', 'sta_assmt_credit',
            'units', 'stories', 'struct_area', 'land_area', 'prior_total_assmt']
for col in NUM_COLS:
    gdf[col] = pd.to_numeric(gdf[col], errors='coerce').fillna(0).clip(lower=0)

gdf['full_market_value'] = gdf['cur_land'] + gdf['cur_impr']

# $0-land-and-$0-improvement accounts (HOA common areas, open-space and right-of-way slivers)
# carry no assessment and never move under any rate change; drop them before classifying.
zero_value = (gdf['full_market_value'] <= 0) & (gdf['cur_total_assmt'] <= 0)
n_zero_value = int(zero_value.sum())
gdf = gdf[~zero_value].copy()
print(f"Dropped {n_zero_value:,} $0-value accounts (common areas and slivers; no assessment)")

gdf['full_exmp'] = ((gdf['cty_exempt_assmt'] > 0)
                    & (gdf['cty_exempt_assmt'] >= gdf['cur_total_assmt'])).astype(int)
partial = (gdf['cty_exempt_assmt'] > 0) & (gdf['full_exmp'] == 0)

print(f"Fully exempt (exempt assessment covers the whole assessment): {int(gdf['full_exmp'].sum()):,}")
print(f"Partially exempt: {int(partial.sum()):,}  (exempt assessment ${gdf.loc[partial, 'cty_exempt_assmt'].sum()/1e6:,.1f}M)")
print(f"Homestead credit recipients: {int((gdf['cty_assmt_credit'] > 0).sum()):,}")
print(f"Assessment above full market value (>0.1%): {int((gdf['cur_total_assmt'] > gdf['full_market_value'] * 1.001).sum()):,}")

Dropped 218 stale records (assessment cycle before 2027); 112,796 current accounts


Dropped 4,069 $0-value accounts (common areas and slivers; no assessment)
Fully exempt (exempt assessment covers the whole assessment): 4,837
Partially exempt: 93  (exempt assessment $331.2M)
Homestead credit recipients: 47,740
Assessment above full market value (>0.1%): 4


In [6]:
def categorize(row):
    lu = str(row.get('land_use') or '')
    units = row.get('units') or 0
    desc = str(row.get('DESCCIUSE') or '').upper()

    if lu.startswith('Town House'):
        return 'Townhome / Rowhouse'
    if lu.startswith('Residential Condominium'):
        return 'Condominium'
    if lu.startswith('Residential (R)'):
        if units >= 5:
            return 'Large Multi-Family (5+ units)'
        if units >= 2:
            return 'Small Multi-Family (2-4 units)'
        return 'Single Family Residential'
    if lu.startswith('Apartments'):
        return 'Small Multi-Family (2-4 units)' if 2 <= units <= 4 else 'Large Multi-Family (5+ units)'
    if lu.startswith('Commercial Condominium'):
        return 'Office / Commercial Condo'
    if lu.startswith('Commercial Residential') or lu.startswith('Residential Commercial'):
        return 'Mixed Use'
    if lu.startswith('Agricultural'):
        return 'Agricultural'
    if lu.startswith('Industrial'):
        return 'Industrial'
    if lu.startswith('Commercial') or lu.startswith('Exempt Commercial') or lu.startswith('Country Club'):
        if 'PARKING' in desc:
            return 'Transportation - Parking'
        if any(k in desc for k in ['HOTEL', 'MOTEL', 'INN']):
            return 'Hotel'
        if 'OFFICE' in desc or 'BANK' in desc or 'MEDICAL' in desc:
            return 'Office / Commercial Condo'
        if any(k in desc for k in ['RETAIL', 'STORE', 'SHOPPING', 'SUPERMARKET', 'RESTAURANT',
                                    'SERVICE STATION', 'CONVENIENCE', 'AUTO']):
            return 'Retail / General Commercial'
        if any(k in desc for k in ['WAREHOUSE', 'INDUSTRIAL', 'MANUFACTUR', 'FLEX', 'DISTRIBUTION']):
            return 'Industrial'
        return 'Other Commercial'
    return 'Other'


gdf['PROPERTY_CATEGORY'] = gdf.apply(categorize, axis=1)
gdf.loc[(gdf['cur_impr'] <= 0), 'PROPERTY_CATEGORY'] = 'Vacant Land'

taxable_view = gdf[gdf['full_exmp'] == 0]
print(taxable_view['PROPERTY_CATEGORY'].value_counts().to_string())
print("\nLargest DESCCIUSE values inside 'Other Commercial':")
print(taxable_view.loc[taxable_view['PROPERTY_CATEGORY'] == 'Other Commercial', 'DESCCIUSE']
      .value_counts(dropna=False).head(15).to_string())

PROPERTY_CATEGORY
Single Family Residential         60599
Townhome / Rowhouse               27074
Condominium                        9147
Vacant Land                        2741
Office / Commercial Condo          1725
Agricultural                        694
Industrial                          665
Retail / General Commercial         560
Other Commercial                    174
Large Multi-Family (5+ units)       165
Small Multi-Family (2-4 units)      143
Mixed Use                           140
Hotel                                35
Transportation - Parking             23
Other                                 5

Largest DESCCIUSE values inside 'Other Commercial':
DESCCIUSE
CARE Day Care Center                         30
OTHER Yard Items                             24
NaN                                          13
HOUSING Apartments                           11
REC Country Club Subject to Use Agreement     9
CARE Life Care Facility                       8
REC Recreation Property        

## Section 4: Current tax model + revenue validation

**Taxable base, built from the billed quantities.** For each parcel:

1. *Phase-in.* `cur_total_assmt` is this year's phased-in assessment. It is spread over land and
   improvement in the proportions of the full-market appraisal: `pf = cur_total_assmt / (cur_land + cur_impr)`.
2. *Partial exemptions.* The county exempt assessment comes off the improvement first, then land
   (the repo's exemption hierarchy).
3. *Homestead credit.* The county Homestead credit caps growth of the whole assessment, so it
   scales land and improvement together by `1 - credit / assessment`.

The result sums, parcel by parcel, to `cur_total_assmt - cty_exempt_assmt - cty_assmt_credit`,
the assessment the county actually bills. Both levies use that base:
`current_tax = base x (1.044 + 0.206) / 100`.

**Under the reform** each parcel keeps its phase-in factor, its exemption dollars and its Homestead
ratio; only the rates change. That is the natural carry-over of a cap stated as a share of
assessment growth, but a statute could apply the cap differently (e.g. separately to land and to
improvements), which would change capped parcels' results.

**Validation targets** (Howard County FY 2027 Approved Budget, *Statement of Assessable Base and
Estimated Collections*, p. 417, and the *Fire & Rescue Tax* fund, p. 380):

- Real-property assessable base, FY 2027 projected: **$73,547,030,000**; county real-property
  revenue **$767,831,000** at $1.044.
- Fire & Rescue fund property taxes: **$153,700,000**. That line includes business personal
  property at $0.515; the personal-property base of $2,022,397,000 contributes roughly $10.4M, so
  the real-property share is roughly $143M. The fire check is therefore against rate x the budget's
  own real-property base ($151.5M), with the fund line reported alongside.

In [7]:
pf = (gdf['cur_total_assmt'] / gdf['full_market_value'].replace(0, np.nan)).fillna(0)
land_a = gdf['cur_land'] * pf
impr_a = gdf['cur_impr'] * pf
# Parcels with an assessment but no land/improvement split (none expected) would be dropped here.
no_split = (gdf['full_market_value'] <= 0) & (gdf['cur_total_assmt'] > 0)
assert no_split.sum() == 0, f"{no_split.sum()} parcels have an assessment but no land/improvement split"

# Partial exemptions: improvement first, then land
ex = gdf['cty_exempt_assmt'].where(gdf['full_exmp'] == 0, 0)
ex_impr = np.minimum(ex, impr_a)
impr_b = impr_a - ex_impr
land_b = (land_a - (ex - ex_impr)).clip(lower=0)

# Homestead credit: scale both components
after_ex = land_b + impr_b
hs_ratio = (1 - gdf['cty_assmt_credit'] / after_ex.replace(0, np.nan)).fillna(1).clip(lower=0)
gdf['homestead_ratio'] = hs_ratio
gdf['taxable_land_value'] = land_b * hs_ratio
gdf['taxable_improvement_value'] = impr_b * hs_ratio
gdf.loc[gdf['full_exmp'] == 1, ['taxable_land_value', 'taxable_improvement_value']] = 0
gdf['taxable_total_value'] = gdf['taxable_land_value'] + gdf['taxable_improvement_value']

# Guard: the modeled base must equal the billed base parcel by parcel
billed = (gdf['cur_total_assmt'] - gdf['cty_exempt_assmt'] - gdf['cty_assmt_credit']).clip(lower=0)
billed = billed.where(gdf['full_exmp'] == 0, 0)
resid = (gdf['taxable_total_value'] - billed).abs()
assert resid.max() < 1.0, f"Base reconstruction off by up to ${resid.max():,.0f}"

gdf['millage_rate'] = COMBINED_MILLAGE
current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='taxable_total_value',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

OFFICIAL_BASE = 73_547_030_000
OFFICIAL_COUNTY_REVENUE = 767_831_000
OFFICIAL_FIRE_RATE_X_BASE = OFFICIAL_BASE * FIRE_MILLAGE / 1000
OFFICIAL_FIRE_FUND_PROPERTY_TAX = 153_700_000
OFFICIAL_REVENUE = OFFICIAL_COUNTY_REVENUE + OFFICIAL_FIRE_RATE_X_BASE

base = float(gdf['taxable_total_value'].sum())
county_rev = base * COUNTY_MILLAGE / 1000
fire_rev = base * FIRE_MILLAGE / 1000
print(f"Modeled taxable base:  ${base/1e9:,.3f}B   vs budget ${OFFICIAL_BASE/1e9:,.3f}B  ({(base/OFFICIAL_BASE-1)*100:+.2f}%)")
print(f"  full market value of taxable parcels ${gdf.loc[gdf['full_exmp']==0, 'full_market_value'].sum()/1e9:,.3f}B;"
      f" phase-in withheld ${(gdf['full_market_value']-gdf['cur_total_assmt']).where(gdf['full_exmp']==0,0).sum()/1e9:,.3f}B;"
      f" Homestead credit ${gdf['cty_assmt_credit'].where(gdf['full_exmp']==0,0).sum()/1e9:,.3f}B")
print(f"County levy:  ${county_rev:,.0f}   vs ${OFFICIAL_COUNTY_REVENUE:,.0f}  ({(county_rev/OFFICIAL_COUNTY_REVENUE-1)*100:+.2f}%)")
print(f"Fire levy:    ${fire_rev:,.0f}   vs rate x budget base ${OFFICIAL_FIRE_RATE_X_BASE:,.0f}  ({(fire_rev/OFFICIAL_FIRE_RATE_X_BASE-1)*100:+.2f}%)"
      f";  vs fund property-tax line ${OFFICIAL_FIRE_FUND_PROPERTY_TAX:,.0f} (incl. personal property)")
print(f"Combined:     ${current_revenue:,.0f}   vs ${OFFICIAL_REVENUE:,.0f}  ({(current_revenue/OFFICIAL_REVENUE-1)*100:+.2f}%)")
gap_pct = (current_revenue / OFFICIAL_REVENUE - 1) * 100
assert abs(gap_pct) < 5.0, f"Revenue gap {gap_pct:.1f}% exceeds threshold"

Modeled taxable base:  $73.880B   vs budget $73.547B  (+0.45%)
  full market value of taxable parcels $77.858B; phase-in withheld $1.347B; Homestead credit $2.300B
County levy:  $771,307,648   vs $767,831,000  (+0.45%)
Fire levy:    $152,192,888   vs rate x budget base $151,506,882  (+0.45%);  vs fund property-tax line $153,700,000 (incl. personal property)
Combined:     $923,500,536   vs $919,337,882  (+0.45%)


## Section 5: Split-rate model (4:1)

Fully-exempt parcels are held out of the solver and dropped from the modeled output (they carry
no signal); their count is kept for the validation summary.

In [8]:
n_exempt = int((gdf['full_exmp'] == 1).sum())
n_zero = int(((gdf['full_exmp'] == 0) & (gdf['taxable_total_value'] <= 0)).sum())
gdf = gdf[(gdf['full_exmp'] == 0) & (gdf['taxable_total_value'] > 0)].copy()

land_millage, improvement_millage, new_revenue, gdf = model_split_rate_tax(
    df=gdf,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=gdf['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

print(f"Held out {n_exempt:,} fully-exempt parcels and {n_zero:,} taxable parcels with a $0 base.")
print(f"Modeled parcels: {len(gdf):,}")
print(f"Land millage:        {land_millage:.4f} per $1,000  (${land_millage/10:.4f}/$100)")
print(f"Improvement millage: {improvement_millage:.4f} per $1,000  (${improvement_millage/10:.4f}/$100)")
print(f"Current combined:    {COMBINED_MILLAGE:.4f} per $1,000")
print(f"Revenue check: ${new_revenue:,.0f} vs current ${gdf['current_tax'].sum():,.0f}")
print(f"Land share of taxable base: {gdf['taxable_land_value'].sum()/gdf['taxable_total_value'].sum()*100:.1f}%")

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title=f"{CITY_NAME}: 4:1 Split-Rate Tax Impact (county + fire)")

Held out 4,837 fully-exempt parcels and 0 taxable parcels with a $0 base.
Modeled parcels: 103,890
Land millage:        24.0258 per $1,000  ($2.4026/$100)
Improvement millage: 6.0065 per $1,000  ($0.6006/$100)
Current combined:    12.5000 per $1,000
Revenue check: $923,500,536 vs current $923,500,536
Land share of taxable base: 36.0%

howard_county: 4:1 Split-Rate Tax Impact (county + fire)


                      Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
     Single Family Residential  60599     $25,796,151        4.8%       $426         $710    9.1%       9.2%            48.0%            12.4%
           Townhome / Rowhouse  27074      $7,239,690        4.9%       $267         $363    6.8%       7.2%            40.2%            10.3%
                   Condominium   9147     $-2,729,149       -8.6%      $-298        $-299   -8.7%      -8.7%             0.2%             1.6%
                   Vacant Land   2741      $5,758,093       92.2%     $2,101         $836   92.2%      92.2%           100.0%             0.0%
     Office / Commercial Condo   1725     $-5,816,950      -16.7%    $-3,372         $229    4.5%       5.7%             7.9%            10.7%
                  Agricultural    694        $341,958        4.6%       $493       $1,004   14.0%      13.3%            55.3%            17.6%

In [9]:
# Artifact scan (validate.md Gate 5)
d = gdf.copy()
d['bldg_share'] = d['taxable_improvement_value'] / d['taxable_total_value'].replace(0, np.nan)
scan = d.groupby('PROPERTY_CATEGORY').agg(
    n=('tax_change_pct', 'size'),
    median_pct=('tax_change_pct', 'median'),
    median_bldg_share=('bldg_share', 'median'),
    p10_bldg_share=('bldg_share', lambda s: s.quantile(0.1)),
    p90_bldg_share=('bldg_share', lambda s: s.quantile(0.9)),
    pct_zero_bldg=('taxable_improvement_value', lambda s: (s <= 1000).mean()),
    pct_homestead=('homestead_ratio', lambda s: (s < 1).mean()),
).sort_values('median_pct', ascending=False)
print(scan.round(3).to_string())
print(f"\nCeiling (all-land parcel): {(land_millage/COMBINED_MILLAGE-1)*100:+.1f}%   "
      f"Floor (all-building parcel): {(improvement_millage/COMBINED_MILLAGE-1)*100:+.1f}%")

                                    n  median_pct  median_bldg_share  p10_bldg_share  p90_bldg_share  pct_zero_bldg  pct_homestead
PROPERTY_CATEGORY                                                                                                                 
Vacant Land                      2741      92.207              0.000           0.000           0.000          1.000          0.000
Transportation - Parking           23      59.037              0.230           0.090           0.317          0.000          0.000
Mixed Use                         140      20.647              0.496           0.265           0.675          0.007          0.143
Small Multi-Family (2-4 units)    143      17.306              0.520           0.310           0.765          0.021          0.385
Agricultural                      694      13.335              0.547           0.339           0.755          0.009          0.519
Single Family Residential       60599       9.248              0.575           0.44

## Section 6: Exploration chart

In [10]:
fig, ax = plt.subplots(figsize=(10, 6))
summary = gdf.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values()
summary.plot.barh(ax=ax, color=['#c0392b' if v > 0 else '#27ae60' for v in summary.values])
ax.axvline(0, color='black', linewidth=0.5)
ax.set_title(f'{CITY_NAME}: Median Tax Change % by Category (4:1 split-rate)')
ax.set_xlabel('Median % Change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=120)
plt.close()
print(f"Saved {DATA_DIR / 'category_preview.png'}")

Saved data\category_preview.png


## Section 7: Census join + standard export

In [11]:
# Census join — must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            # census_gdf already carries demographics — spatial join adds them above.
            # Do NOT do a second gdf.merge(census_data) here: census_gdf has the columns
            # baked in, so a second merge creates median_income_x/y duplicates and silently
            # zeros out all demographic output.
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print("Census API timed out — skipping census join")
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

Census join: 99.3% matched


In [12]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
    parcel_id_col=PARCEL_ID_COL,
)

# Standard report — 7 PNGs in analysis/reports/<city>/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)

# Parcel-map export — GeoParquet (geometry + tax outcomes + parcel id/owner/address) and an
# interactive HTML map colored by tax change.
from lvt.parcel_map import save_parcel_map_export, create_parcel_map
map_gdf = save_parcel_map_export(
    gdf=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/maps/{CITY_NAME}.parquet',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    parcel_id_col=PARCEL_ID_COL,
    parcel_url_template=PARCEL_URL_TEMPLATE,
    owner_name_col=OWNER_NAME_COL,
    owner_address_col=OWNER_ADDRESS_COL,
)
# SDAT record links: the detail URL needs district + account separately, which a {parcel_id}
# template cannot build, so attach the per-account link built in Section 2 and rewrite the parquet.
_links = gdf.drop_duplicates('acctid').set_index('acctid')['SDATWEBADR']
map_gdf['parcel_url'] = map_gdf['parcel_id'].map(_links).where(lambda s: s.notna() & (s != ''), None)
map_gdf.to_parquet(f'../../analysis/maps/{CITY_NAME}.parquet')
print(f"SDAT links attached: {map_gdf['parcel_url'].notna().sum():,} of {len(map_gdf):,}")
create_parcel_map(map_gdf, CITY_NAME)
print("Done.")

  ✓ howard_county: 103,890 rows → ../../analysis/data/howard_county.csv  [model: split_rate:4.0]


  ✓ howard_county: 103,175 parcels → ../../analysis/maps/howard_county.parquet  [map export; 0 with record links]


SDAT links attached: 103,175 of 103,175
  [warn] howard_county: 103,175 parcels exceeds tile_threshold (100,000) but tippecanoe is not installed — falling back to the inline Leaflet map (this file will be large).


  ✓ howard_county: interactive map → ../../analysis/reports\howard_county\parcel_map.html  [103,175 parcels]
Done.


## Validation summary

Printed from this run's values rather than typed, so it cannot drift from the model.

In [13]:
_csv = pd.read_csv(f'../../analysis/data/{CITY_NAME}.csv')
_pngs = sorted(p.name for p in (REPO_ROOT / 'analysis' / 'reports' / CITY_NAME).glob('*.png'))
_med = gdf.groupby('PROPERTY_CATEGORY')['tax_change_pct'].median()
rows = [
    ('Current roll accounts (cycle 2027)', f"{n_current:,}"),
    ('Dropped: $0-value accounts', f"{n_zero_value:,}"),
    ('Held out: fully exempt', f"{n_exempt:,}"),
    ('Modeled taxable accounts', f"{len(gdf):,}"),
    ('Taxable base vs FY27 budget', f"${base/1e9:,.3f}B vs ${OFFICIAL_BASE/1e9:,.3f}B ({(base/OFFICIAL_BASE-1)*100:+.2f}%)"),
    ('County levy vs FY27 budget', f"${county_rev/1e6:,.1f}M vs ${OFFICIAL_COUNTY_REVENUE/1e6:,.1f}M ({(county_rev/OFFICIAL_COUNTY_REVENUE-1)*100:+.2f}%)"),
    ('Fire levy vs rate x budget base', f"${fire_rev/1e6:,.1f}M vs ${OFFICIAL_FIRE_RATE_X_BASE/1e6:,.1f}M ({(fire_rev/OFFICIAL_FIRE_RATE_X_BASE-1)*100:+.2f}%)"),
    ('Land / improvement millage (4:1)', f"{land_millage:.3f} / {improvement_millage:.3f} per $1,000 (current {COMBINED_MILLAGE:.3f})"),
    ('Census coverage (export)', f"{_csv['std_geoid'].notna().mean()*100:.1f}%"),
    ('Report PNGs', f"{len(_pngs)}"),
    ('Map parcels (with geometry) / SDAT links', f"{len(map_gdf):,} / {map_gdf['parcel_url'].notna().sum():,}"),
    ('SFR / Townhome / Condo median change', f"{_med.get('Single Family Residential', np.nan):+.1f}% / {_med.get('Townhome / Rowhouse', np.nan):+.1f}% / {_med.get('Condominium', np.nan):+.1f}%"),
    ('Large MF / Vacant Land median change', f"{_med.get('Large Multi-Family (5+ units)', np.nan):+.1f}% / {_med.get('Vacant Land', np.nan):+.1f}%"),
]
print(pd.DataFrame(rows, columns=['Check', 'Result']).to_string(index=False))

                                   Check                                     Result
      Current roll accounts (cycle 2027)                                    112,796
              Dropped: $0-value accounts                                      4,069
                  Held out: fully exempt                                      4,837
                Modeled taxable accounts                                    103,890
             Taxable base vs FY27 budget              $73.880B vs $73.547B (+0.45%)
              County levy vs FY27 budget                $771.3M vs $767.8M (+0.45%)
         Fire levy vs rate x budget base                $152.2M vs $151.5M (+0.45%)
        Land / improvement millage (4:1) 24.026 / 6.006 per $1,000 (current 12.500)
                Census coverage (export)                                      99.3%
                             Report PNGs                                          7
Map parcels (with geometry) / SDAT links                          103,175 / 

### Reading the results

- **Detached homes and townhouses rise; apartments, hotels, industrial and condos fall.** A split
  rate moves tax toward parcels whose land share exceeds the base-wide land share. Howard's
  detached homes sit on larger lots than that average, while apartment complexes, industrial
  buildings and hotels are building-heavy, so the direction follows from the county's composition,
  as in Rockville. It is not the city pattern `validate.md` expects, where homes fall.
- **Vacant land sits exactly at the ceiling** (`land_millage / current_millage - 1`) because it is
  all land. The zero-improvement commercial and industrial lots were checked against CAMA structure
  area, which is recorded for ~94% of built commercial accounts and is zero on these: they are
  unbuilt pads (Columbia Gateway, the Merriweather district), not placeholder values.
- **Homestead parcels.** The credit scales both components together, so a capped home's percentage
  change matches an uncapped home with the same land share; the cap changes dollars, not direction.

### Limitations

- **Condominium land shares are an assessor convention.** SDAT books every residential condo at
  exactly 30% land and every commercial condo at 40% (the 10th and 90th percentiles coincide).
  Condo results therefore reflect that convention, not a measured per-unit land value; a condo
  unit's true share of its building's land is usually well below 30% in mid-rise buildings and
  near it in garden-style ones.
- **Agricultural land** enters at its preferential use-value assessment (it is already inside
  `cur_land`), so farms are modeled at use value, as they are billed.
- **Fire fund line.** The Fire & Rescue fund budgets property taxes about 4% below rate x base
  (after removing the personal-property share) in both FY 2026 and FY 2027. The county levy, which
  shares the base, reconciles within 0.5%, so the gap reads as a collection or credit allowance in
  the fund budget rather than a base difference; it does not affect percentage changes.
- **Not modeled**: the state property tax ($0.112), the Metropolitan District ad valorem charge,
  business personal property, and the two TIF districts' increment (billed at the same rates; the
  county's share of the increment is diverted to the TIF funds, which does not change bills).
- **Geometry**: accounts without their own iMAP polygon (chiefly condo units and newer townhouses)
  are drawn as small circles at the roll's point location; accounts with no point either stay in
  the CSV export but are absent from the map.